In [ ]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
from ipywidgets import widgets

from latent_dynamics.dayang.activations import Activations, extract_activations
from latent_dynamics.dayang.data import DATASET_REGISTRY, load_dataset_from_spec
from latent_dynamics.dayang.model import (
    MODEL_REGISTRY,
    get_token_embeddings,
    get_token_groups,
    load_model_and_tokenizer,
    print_token_groups,
)
from latent_dynamics.dayang.projections import (
    compute_pca_per_layer,
    compute_pca_per_token,
    plot_pca_per_layer,
    plot_pca_per_token,
    plot_pca_explained_variance,
    plot_token_embeddings,
)
from latent_dynamics.dayang.scoring import compute_score_per_layer, plot_score_per_layer
from latent_dynamics.dayang.utils import select
from latent_dynamics.dayang.widgets import (
    ActivationsExtractorWidget,
    ActivationsSelectorWidget,
    TokenEmbeddingsLoaderWidget,
)

torch.set_grad_enabled(False)

print(f"Available models:{'\n - '.join([''] + list(MODEL_REGISTRY))}")
print(f"Available datasets:{'\n - '.join([''] + list(DATASET_REGISTRY))}")

# Manual analysis

In [ ]:
dataset = load_dataset_from_spec("xstest", max_samples=200)

In [ ]:
model, tokenizer = load_model_and_tokenizer("gemma3_270m")

## Generate responses

In [ ]:
from latent_dynamics.dayang.activations import _prepare_sample


def generate(model, tokenizer, sample, include_response, apply_chat_template):
    sample_prepared = _prepare_sample(
        sample, tokenizer, include_response=include_response, apply_chat_template=apply_chat_template
    )
    inputs = tokenizer(sample_prepared["input"], return_tensors="pt", add_special_tokens=not apply_chat_template)
    outputs = model.generate(**inputs, max_new_tokens=10)
    response = tokenizer.decode(outputs)[0]
    return response


def test_generate(model, tokenizer, sample, include_responses=[False, "Sure", "Sorry"], apply_chat_template=True):
    print(f"ID:             {sample['id']}")
    print(f"is_safe:        {sample['is_safe']}")
    print(f"is_adversarial: {sample['is_adversarial']}")
    for include_response in include_responses:
        print("-" * 50)
        response = generate(
            model,
            tokenizer,
            sample,
            include_response=include_response,
            apply_chat_template=apply_chat_template,
        )
        print(response)


test_generate(model, tokenizer, dataset[0], apply_chat_template=True)

## Extract activations

In [ ]:
activations = extract_activations(model, tokenizer, dataset, apply_chat_template=True)
activations.samples

In [ ]:
activations = Activations.load("../activations/xstest/gemma3_270m/chat")
activations.samples

In [ ]:
activations.extract_topk(model, tokenizer, layer=18)

## Token embedding analysis

In [ ]:
token_groups = get_token_groups(tokenizer)
print_token_groups(tokenizer, token_groups)

In [ ]:
plot_token_embeddings(
    model,
    tokenizer,
    token_groups,
    num_components=8,
)

## PCA analysis per layer

In [ ]:
pcas = compute_pca_per_layer(activations, pool_method="last")
plot_pca_explained_variance(pcas)

In [ ]:
plot_pca_per_layer(
    activations.select(activations.samples_safe.index[:3].tolist() + activations.samples_unsafe.index[:3].tolist()),
    pcas,
    pool_method=slice(-6, None),
    exclude_special_tokens=False,
)

## PCA analysis per token

In [ ]:
pcas = compute_pca_per_token(
    activations.select([activations.samples_safe.index[0], activations.samples_unsafe.index[0]]),
    pool_method=slice(-8, None),
    separate=True,
)
plot_pca_explained_variance(pcas)

In [ ]:
plot_pca_per_token(
    activations.select([activations.samples_safe.index[0], activations.samples_unsafe.index[0]]),
    pcas,
    pool_method=slice(-8, None),
    separate=True,
    token_embeddings=get_token_embeddings(model, tokenizer, select(range(tokenizer.vocab_size), at_most=0.25)),
)

## Scoring analysis per layer

In [ ]:
scorers = compute_score_per_layer(activations, method="difference_in_mean")

In [ ]:
plot_score_per_layer(activations, scorers, pool_method="last")

# Interactive analysis

### PCA analysis per layer

In [ ]:
def analyze_pca_per_layer():
    state = {"models": {}}

    # Define layer 1 widgets for extracting activations
    out_layer1 = widgets.Output()
    w_act_extractor = ActivationsExtractorWidget(
        out=out_layer1,
        search_path="../activations",
        dataset="xstest",
    )
    w_emb_loader = TokenEmbeddingsLoaderWidget(
        out=out_layer1,
    )
    layer1 = widgets.VBox([w_act_extractor, w_emb_loader, out_layer1])

    # Define layer 2 widgets for analyzing activations
    w_train_selector = ActivationsSelectorWidget()
    w_eval_selector = ActivationsSelectorWidget()
    w_eval_apply = widgets.Dropdown(options=["per layer"], value="per layer", description="Apply")
    btn_analyze = widgets.Button(description="Analyze", button_style="primary", disabled=True)
    out_analyze = widgets.Output()

    col_train = widgets.VBox([widgets.HTML("<b>PCA</b>"), w_train_selector])
    col_eval = widgets.VBox([widgets.HTML("<b>Visualization</b>"), w_eval_selector, w_eval_apply])
    layer2 = widgets.VBox(
        [
            widgets.HBox([col_train, col_eval]),
            btn_analyze,
            out_analyze,
        ]
    )

    # Register handlers for layer 1
    def update_layer2_widgets(*args):
        activations = w_act_extractor.activations

        # Update state
        state["models"] = {}

        # Update widgets
        if activations is None:
            w_eval_apply.options = ["per layer"]
            btn_analyze.disabled = True
        else:
            w_train_selector.set_activations(activations)
            w_eval_selector.set_activations(activations)
            w_eval_apply.options = ["per layer"] + list(range(activations.num_layers))
            btn_analyze.disabled = False
        out_analyze.clear_output()

    w_act_extractor.observe(update_layer2_widgets, names="value")

    # Register handlers for layer 2
    def do_plot(*args):
        activations = w_act_extractor.activations
        if activations is None:
            return

        with out_analyze:
            out_analyze.clear_output(wait=True)

            # Select samples training
            train_sample_ids = w_train_selector.samples
            train_activations = activations.select(train_sample_ids) if train_sample_ids else activations
            # Train and cache model
            cache_key = (
                tuple(sorted(train_sample_ids)),
                tuple(w_train_selector.pool_method) if isinstance(w_train_selector.pool_method, list) else w_train_selector.pool_method,
                w_train_selector.exclude_bos,
                w_train_selector.exclude_special_tokens,
            )
            if cache_key not in state["models"]:
                state["models"][cache_key] = compute_pca_per_layer(
                    train_activations,
                    pool_method=w_train_selector.pool_method,
                    exclude_bos=w_train_selector.exclude_bos,
                    exclude_special_tokens=w_train_selector.exclude_special_tokens,
                )
            # Plot PCA explained variance ratio
            plot_pca_explained_variance(state["models"][cache_key])

            # Select samples for evaluation
            eval_sample_ids = w_eval_selector.samples
            eval_activations = activations.select(eval_sample_ids) if eval_sample_ids else activations
            # Plot evaluation
            if w_eval_apply.value == "per layer":
                models = state["models"][cache_key]
            else:
                models = [state["models"][cache_key][w_eval_apply.value]] * activations.num_layers
            plot_pca_per_layer(
                eval_activations,
                models,
                pool_method=w_eval_selector.pool_method,
                exclude_bos=w_eval_selector.exclude_bos,
                exclude_special_tokens=w_eval_selector.exclude_special_tokens,
                token_embeddings=w_emb_loader.token_embeddings,
            )

    btn_analyze.on_click(do_plot)

    # Display widgets
    display(widgets.VBox([layer1, widgets.HTML("<hr>"), layer2]))


analyze_pca_per_layer()

### PCA analysis per token

In [ ]:
def analyze_pca_per_token():
    state = {"models": {}}

    # Define layer 1 widgets for extracting activations
    out_layer1 = widgets.Output()
    w_act_extractor = ActivationsExtractorWidget(
        out=out_layer1,
        search_path="../activations",
        dataset="xstest",
    )
    w_emb_loader = TokenEmbeddingsLoaderWidget(
        out=out_layer1,
    )
    layer1 = widgets.VBox([w_act_extractor, w_emb_loader, out_layer1])

    # Define layer 2 widgets for analyzing activations
    w_train_selector = ActivationsSelectorWidget(
        pool_method="all",
    )
    w_eval_selector = ActivationsSelectorWidget(
        pool_method="all",
    )
    w_eval_separate = widgets.Checkbox(value=False, description="Separate plots")
    btn_analyze = widgets.Button(description="Analyze", button_style="primary", disabled=True)
    out_analyze = widgets.Output()

    col_train = widgets.VBox([widgets.HTML("<b>PCA</b>"), w_train_selector])
    col_eval = widgets.VBox([widgets.HTML("<b>Visualization</b>"), w_eval_selector, w_eval_separate])
    layer2 = widgets.VBox(
        [
            widgets.HBox([col_train, col_eval]),
            btn_analyze,
            out_analyze,
        ]
    )

    # Register handlers for layer 1
    def update_layer2_widgets(*args):
        activations = w_act_extractor.activations

        # Update state
        state["models"] = {}

        # Update widgets
        if activations is None:
            btn_analyze.disabled = True
        else:
            w_train_selector.set_activations(activations)
            w_train_selector.w_safe.index = (0,)
            w_eval_selector.set_activations(activations)
            w_eval_selector.w_safe.index = (0,)
            btn_analyze.disabled = False
        out_analyze.clear_output()

    w_act_extractor.observe(update_layer2_widgets, names="value")

    # Register handlers for layer 2
    def do_plot(*args):
        activations = w_act_extractor.activations
        if activations is None:
            return

        with out_analyze:
            out_analyze.clear_output(wait=True)

            # Select samples training
            train_sample_ids = w_train_selector.samples
            train_activations = activations.select(train_sample_ids) if train_sample_ids else activations
            # Train and cache model
            cache_key = (
                tuple(sorted(train_sample_ids)),
                tuple(w_train_selector.pool_method) if isinstance(w_train_selector.pool_method, list) else w_train_selector.pool_method,
                w_train_selector.exclude_bos,
                w_train_selector.exclude_special_tokens,
            )
            if cache_key not in state["models"]:
                state["models"][cache_key] = compute_pca_per_token(
                    train_activations,
                    pool_method=w_train_selector.pool_method,
                    exclude_bos=w_train_selector.exclude_bos,
                    exclude_special_tokens=w_train_selector.exclude_special_tokens,
                )
            # Plot PCA explained variance ratio
            plot_pca_explained_variance(state["models"][cache_key])

            # Select samples for evaluation
            eval_sample_ids = w_eval_selector.samples
            eval_activations = activations.select(eval_sample_ids) if eval_sample_ids else activations
            # Plot evaluation
            plot_pca_per_token(
                eval_activations,
                state["models"][cache_key],
                pool_method=w_eval_selector.pool_method,
                exclude_bos=w_eval_selector.exclude_bos,
                exclude_special_tokens=w_eval_selector.exclude_special_tokens,
                token_embeddings=w_emb_loader.token_embeddings,
                separate=w_eval_separate.value,
            )

    btn_analyze.on_click(do_plot)

    # Display widgets
    display(widgets.VBox([layer1, widgets.HTML("<hr>"), layer2]))


analyze_pca_per_token()

### Scoring analysis per layer

In [ ]:
def analyze_score_per_layer():
    state = {"models": {}}

    # Define layer 1 widgets for extracting activations
    layer1 = ActivationsExtractorWidget(
        search_path="../activations",
        dataset="xstest",
    )

    # Define layer 2 widgets for analyzing activations
    w_train_selector = ActivationsSelectorWidget()
    w_eval_selector = ActivationsSelectorWidget()
    w_eval_apply = widgets.Dropdown(options=["per layer"], value="per layer", description="Apply")
    btn_analyze = widgets.Button(description="Analyze", button_style="primary", disabled=True)
    out_analyze = widgets.Output()

    w_method = widgets.Dropdown(
        options=["difference_in_mean", "linear_probe"], value="difference_in_mean", description="Method"
    )
    col_train = widgets.VBox([widgets.HTML("<b>Train</b>"), w_train_selector])
    col_eval = widgets.VBox([widgets.HTML("<b>Visualization</b>"), w_eval_selector, w_eval_apply])
    layer2 = widgets.VBox(
        [
            w_method,
            widgets.HBox([col_train, col_eval]),
            btn_analyze,
            out_analyze,
        ]
    )

    # Register handlers for layer 1
    def update_layer2_widgets(*args):
        activations = layer1.activations

        # Update state
        state["models"] = {}

        # Update widgets
        if activations is None:
            w_eval_apply.options = ["per layer"]
            btn_analyze.disabled = True
        else:
            w_train_selector.set_activations(activations)
            w_eval_selector.set_activations(activations)
            w_eval_apply.options = ["per layer"] + list(range(activations.num_layers))
            btn_analyze.disabled = False
        out_analyze.clear_output()

    layer1.observe(update_layer2_widgets, names="value")

    # Register handlers for layer 2
    def do_plot(*args):
        activations = layer1.activations
        if activations is None:
            return

        with out_analyze:
            out_analyze.clear_output(wait=True)

            # Select samples training
            train_sample_ids = w_train_selector.samples
            train_activations = activations.select(train_sample_ids) if train_sample_ids else activations
            # Train and cache model
            cache_key = (
                tuple(sorted(train_sample_ids)),
                w_method.value,
                tuple(w_train_selector.pool_method) if isinstance(w_train_selector.pool_method, list) else w_train_selector.pool_method,
                w_train_selector.exclude_bos,
                w_train_selector.exclude_special_tokens,
            )
            if cache_key not in state["models"]:
                state["models"][cache_key] = compute_score_per_layer(
                    train_activations,
                    method=w_method.value,
                    pool_method=w_train_selector.pool_method,
                    exclude_bos=w_train_selector.exclude_bos,
                    exclude_special_tokens=w_train_selector.exclude_special_tokens,
                )

            # Select samples for evaluation
            eval_sample_ids = w_eval_selector.samples
            eval_activations = activations.select(eval_sample_ids) if eval_sample_ids else activations
            # Plot evaluation
            if w_eval_apply.value == "per layer":
                models = state["models"][cache_key]
            else:
                models = [state["models"][cache_key][w_eval_apply.value]] * activations.num_layers
            plot_score_per_layer(
                eval_activations,
                models,
                pool_method=w_eval_selector.pool_method,
                exclude_bos=w_eval_selector.exclude_bos,
                exclude_special_tokens=w_eval_selector.exclude_special_tokens,
            )

    btn_analyze.on_click(do_plot)

    # Display widgets
    display(widgets.VBox([layer1, widgets.HTML("<hr>"), layer2]))


analyze_score_per_layer()